# Gravitational pull among bodies in a plane

Each of `n` bodies of unit mass lies at a point in a plane and is pulled by every other body.
The pull of body `j` on body `i` is the vector from `i` to `j`, divided by the cube of a
softened distance. The square of the softened distance is the square of the true distance plus
the small constant 0.0025, which is the square of 0.05. The softening keeps the pull finite when
two bodies come close. The acceleration of a body is the sum of the pulls on it. Every body is
compared with every other, so the work grows as the square of the number of bodies. The function
below handles a block of bodies at a time, so that it never holds an array of all the pairs. The
demonstration prints the accelerations of three bodies at the corners of a triangle.

In [ ]:
import numpy as np

SOFTENING_SQUARED = 0.0025  # the square of the softening length, 0.05


def nbody_accel(px, py):
    """Accelerations (ax, ay) of unit-mass bodies at (px, py) under softened gravity."""
    ax, ay = np.empty_like(px), np.empty_like(py)
    for start in range(0, len(px), 256):
        rows = slice(start, start + 256)
        dx = px[None, :] - px[rows, None]  # dx[i, j] is the x distance from body i to body j
        dy = py[None, :] - py[rows, None]
        r2 = dx * dx + dy * dy + SOFTENING_SQUARED
        d3 = r2 * np.sqrt(r2)
        ax[rows] = np.sum(dx / d3, axis=1)  # a body's pull on itself is zero, as dx is zero
        ay[rows] = np.sum(dy / d3, axis=1)
    return ax, ay

In [ ]:
print(nbody_accel(np.array([0.0, 1.0, 0.0]), np.array([0.0, 0.0, 1.0])))

The declaration below tells the harness how to call the function and how to check a routine
against it. `make_inputs` places the bodies at random in a square from -1 to 1 in each
direction. A few of them are moved to exactly -1 or 0 in x, and a few more are moved on top of
another body. The tolerance is 1e-13 of the largest acceleration, which is about eight times the
difference of 1.2e-14 between the sums that NumPy computes and the sums that a plain C loop
computes for 16,000 bodies.

In [ ]:
def make_inputs(size, rng):
    px = rng.uniform(-1.0, 1.0, size)
    py = rng.uniform(-1.0, 1.0, size)
    edge = rng.random(size) < 0.05
    px[edge] = rng.choice([-1.0, 0.0], edge.sum())
    twin = rng.random(size) < 0.02
    source = rng.integers(0, size, twin.sum())
    px[twin], py[twin] = px[source], py[source]
    return {"px": px, "py": py}


ROUTINE = {
    "function": nbody_accel,
    "make_inputs": make_inputs,
    "arguments": {"px": "float64[n]", "py": "float64[n]"},
    "results": {"ax": "float64[n]", "ay": "float64[n]"},
    "test_sizes": [1, 30, 300, 2000],
    "benchmark_size": 16_000,
    "tolerance": 1e-13,
}